# 🔬  AI Data Concierge - Reproducible Analysis

<a href="https://colab.research.google.com/" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

---

## 📋 Query
> **What is the distribution of property values across Pittsburgh neighborhoods?**

## 📊 Metadata
| Property | Value |
|----------|-------|
| **Generated** | 2026-05-12 16:26:23 |
| **Data Source** | WPRDC |
| **Sources Used** | Western PA Regional Data Center (WPRDC) |
| **Notebook Version** | 1.0 (Colab Compatible) |

---

## 📖 How to Use This Notebook

This notebook reproduces the exact analysis performed by the ** AI Data Concierge**.
Follow the steps below to verify, modify, or extend the analysis.

### ✅ Quick Start
1. **Run All Cells**: Click `Runtime` → `Run all` (or press `Ctrl+F9`)
2. **Wait for Setup**: The first cells install dependencies and configure the environment

### 🔧 What You Can Do
| Action | Description |
|--------|-------------|
| **Verify** | Run all cells to confirm the original results |
| **Modify** | Change parameters (dates, locations, filters) and re-run |
| **Extend** | Add your own analysis cells below the results |
| **Export** | Download results as CSV, or save notebook to Drive |

### 📚 Notebook Structure
1. **Setup** - Install dependencies (runs once in Colab)
2. **Configuration** - Import libraries and set up API connections
3. **Data Retrieval** - Fetch data from the data source
4. **Analysis** - Process and analyze the data
5. **Results** - View the final answer and confidence scores
6. **Citations** - Reference sources for your research

---


In [ ]:
# ============================================================
# STEP 1: Environment Setup
# ============================================================
# This cell installs all required packages for Google Colab.
# If running locally, you can skip this cell if packages are installed.

# Check if running in Google Colab
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("🔧 Running in Google Colab - Installing dependencies...")
    !pip install -q pandas numpy matplotlib seaborn requests
    print("✅ Dependencies installed successfully!")
else:
    print("💻 Running locally - assuming dependencies are installed")
    print("   If not, run: pip install pandas numpy matplotlib seaborn requests")


In [ ]:
# ============================================================
# STEP 2: Import Libraries
# ============================================================
# These are the core libraries used throughout this notebook.
# Each import serves a specific purpose in the data pipeline.

import requests      # For making HTTP requests to data APIs
import pandas as pd  # For data manipulation and analysis
import numpy as np   # For numerical computations
from datetime import datetime  # For timestamp handling
import json          # For JSON parsing

# Visualization libraries (with graceful fallback)
try:
    import matplotlib.pyplot as plt  # For creating plots
    import seaborn as sns            # For statistical visualizations
    VISUALIZATION_AVAILABLE = True
    plt.style.use('seaborn-v0_8-whitegrid')
    print("📊 Visualization libraries loaded successfully")
except ImportError:
    VISUALIZATION_AVAILABLE = False
    print("⚠️ Visualization libraries not available")
    print("   Install with: pip install matplotlib seaborn")

# ============================================================
# STEP 3: Configuration
# ============================================================
# Data source configuration - modify these if needed

CKAN_URL = "https://data.wprdc.org"

# Display configuration info
print(f"\n📅 Notebook generated: {datetime.now().isoformat()}")
print(f"🔗 CKAN URL: {CKAN_URL}")
print(f"📌 Timestamp: 2026-05-12T16:26:23.121325")


In [ ]:
# ============================================================
# STEP 4: Helper Functions
# ============================================================
# These utility functions handle data fetching from the CKAN API.
# You can reuse these functions for your own data exploration.

def fetch_ckan_data(resource_id: str, limit: int = 10000, filters: dict = None) -> pd.DataFrame:
    """
    Fetch data from CKAN DataStore API.

    This function handles pagination automatically and returns all records
    up to the specified limit.

    Parameters:
    -----------
    resource_id : str
        The unique identifier for the CKAN resource (dataset)
    limit : int, default=10000
        Maximum number of records to fetch
    filters : dict, optional
        Field filters to apply (e.g., {"state": "CA"})

    Returns:
    --------
    pd.DataFrame
        A DataFrame containing the fetched records

    Example:
    --------
    >>> df = fetch_ckan_data("abc123", limit=1000, filters={"year": 2023})
    >>> print(f"Loaded {len(df)} records")
    """
    print(f"📥 Fetching data from resource: {resource_id}")

    all_records = []
    offset = 0
    batch_size = min(32000, limit)

    while offset < limit:
        params = {
            "resource_id": resource_id,
            "limit": min(batch_size, limit - offset),
            "offset": offset,
        }

        if filters:
            params["filters"] = filters

        response = requests.post(
            f"{CKAN_URL}/api/3/action/datastore_search",
            json=params,
            headers={"Content-Type": "application/json"},
        )

        if response.status_code != 200:
            print(f"❌ Error: {response.status_code} - {response.text}")
            break

        result = response.json()
        if not result.get("success"):
            print(f"❌ CKAN error: {result.get('error')}")
            break

        records = result.get("result", {}).get("records", [])
        if not records:
            break

        all_records.extend(records)
        offset += len(records)

        total = result.get("result", {}).get("total", 0)
        print(f"   Progress: {len(all_records):,} / {total:,} records")

        if offset >= total:
            break

    df = pd.DataFrame(all_records)

    # Remove CKAN internal fields that aren't useful for analysis
    internal_cols = ["_id", "_full_text"]
    df = df.drop(columns=[c for c in internal_cols if c in df.columns], errors="ignore")

    print(f"✅ Loaded {len(df):,} records with {len(df.columns)} columns")
    return df


def search_ckan_packages(query: str, rows: int = 10) -> list:
    """
    Search CKAN for datasets (packages) matching a query.

    Parameters:
    -----------
    query : str
        The search query (e.g., "employment statistics")
    rows : int, default=10
        Maximum number of results to return

    Returns:
    --------
    list
        A list of matching package dictionaries

    Example:
    --------
    >>> packages = search_ckan_packages("housing prices", rows=5)
    >>> for pkg in packages:
    ...     print(pkg['title'])
    """
    print(f"🔍 Searching for: '{query}'")

    response = requests.post(
        f"{CKAN_URL}/api/3/action/package_search",
        json={"q": query, "rows": rows},
        headers={"Content-Type": "application/json"},
    )

    if response.status_code != 200:
        print(f"❌ Search failed: {response.status_code}")
        return []

    result = response.json()
    if not result.get("success"):
        print(f"❌ Search error: {result.get('error')}")
        return []

    packages = result.get("result", {}).get("results", [])
    print(f"✅ Found {len(packages)} matching datasets")
    return packages


print("✅ Helper functions loaded successfully!")
print("   - fetch_ckan_data(resource_id, limit, filters)")
print("   - search_ckan_packages(query, rows)")


# Data Analysis Pipeline

The following steps show how the data was searched, loaded, and analyzed. Each step includes executable code you can modify and re-run.

---

## Step 1: Search for Datasets

**Search query:** `property values assessments`
**Organization:** `city-of-pittsburgh`

**Result preview:**
```
Found 5 datasets matching 'property values assessments'

1. **Condemned and Dead-End Properties**
   ID: `condemned-properties`
   This dataset contains condemned properties in the City of Pittsburgh and is maintained by the Department of Permits, Licenses, and Inspections (PLI). Some of the properties are lab
   - Condemned Properties (CSV) [DataStore] ID: `0a963f26-eb4b-4325-bbbc-3ddf6a871410`
   - Condemned Properties (GeoJSON) (GeoJSON) ID: `e8f6f782-de70-4be0-a3fe-33153af8246d`

2. **Pittsburgh American Community Survey 2014 - Miscellaneous Data**
   ID: `pittsburgh-american-community-sur
```


In [ ]:
# Step 1: Search for Datasets

# Search for datasets
import requests, json

params = {"q": 'property values assessments', "rows": 10, "fq": "organization:city-of-pittsburgh"}
resp = requests.get("https://data.wprdc.org/api/3/action/package_search", params=params)
results = resp.json()["result"]["results"]

print(f"Found {resp.json()['result']['count']} datasets")
for i, ds in enumerate(results, 1):
    print(f"\n{i}. {ds['title']}")
    print(f"   ID: {ds['name']}")
    for r in ds.get("resources", []):
        print(f"   - {r['name']} ({r['format']}) ID: {r['id']}")


## Step 2: Search for Datasets

**Search query:** `property assessment parcel value neighborhood`

**Result preview:**
```
Found 9 datasets matching 'property assessment parcel value neighborhood'

1. **Allegheny County Property Assessments**
   ID: `property-assessments`
   Real Property parcel characteristics for Allegheny County, PA. Includes information pertaining to land, values, sales, abatements, and building characteristics (if residential) by 
   - Property Assessments Parcel Data (for downloads) (CSV) [DataStore] ID: `9a1c60bd-f9f7-4aba-aeb7-af8c3aaa44e5`
   - Property Assessments Parcel Data (API version) () [DataStore] ID: `65855e14-549e-4992-b5be-d629afc676fa`
   - Property Dashboard (HTML) ID: `2fae1
```


In [ ]:
# Step 2: Search for Datasets

# Search for datasets
import requests, json

params = {"q": 'property assessment parcel value neighborhood', "rows": 10}
resp = requests.get("https://data.wprdc.org/api/3/action/package_search", params=params)
results = resp.json()["result"]["results"]

print(f"Found {resp.json()['result']['count']} datasets")
for i, ds in enumerate(results, 1):
    print(f"\n{i}. {ds['title']}")
    print(f"   ID: {ds['name']}")
    for r in ds.get("resources", []):
        print(f"   - {r['name']} ({r['format']}) ID: {r['id']}")


## Step 3: Load Data from Resource

**Resource ID:** `65855e14-549e-4992-b5be-d629afc676fa`
**Limit:** 5

**Result preview:**
```
Resource: 65855e14-549e-4992-b5be-d629afc676fa
Total records: 584,905
Loaded: 5
Fields (86): PARID, PROPERTYHOUSENUM, PROPERTYFRACTION, PROPERTYADDRESS, PROPERTYCITY, PROPERTYSTATE, PROPERTYUNIT, PROPERTYZIP, MUNICODE, MUNIDESC, SCHOOLCODE, SCHOOLDESC, LEGAL1, LEGAL2, LEGAL3, NEIGHCODE, NEIGHDESC, TAXCODE, TAXDESC, TAXSUBCODE, TAXSUBCODE_DESC, OWNERCODE, OWNERDESC, CLASS, CLASSDESC, USECODE, USEDESC, LOTAREA, HOMESTEADFLAG, CLEANGREEN, FARMSTEADFLAG, ABATEMENTFLAG, RECORDDATE, SALEDATE, SALEPRICE, SALECODE, SALEDESC, DEEDBOOK, DEEDPAGE, PREVSALEDATE, PREVSALEPRICE, PREVSALEDATE2, PREVSALEPRICE
```


In [ ]:
# Step 3: Load Data from Resource

# Load resource data
import pandas as pd

params = {"resource_id": '65855e14-549e-4992-b5be-d629afc676fa', "limit": 5}
resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search", json=params)
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
print(f"Total available: {result['total']:,}")
print(f"\nColumns: {\", \".join(df.columns.tolist())}")
df.head(10)


## Step 4: SQL Analysis Query

**SQL:**
```sql

SELECT 
  "NEIGHDESC" AS neighborhood,
  COUNT(*) AS total_parcels,
  ROUND(AVG("FAIRMARKETTOTAL")::numeric, 0) AS avg_fair_market_value,
  ROUND(MIN("FAIRMARKETTOTAL")::numeric, 0) AS min_value,
  ROUND(MAX("FAIRMARKETTOTAL")::numeric, 0) AS max_value,
  ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY "FAIRMARKETTOTAL")::numeric, 0) AS median_value,
  SUM("FAIRMARKETTOTAL") AS total_value
FROM "65855e14-549e-4992-b5be-d629afc676fa"
WHERE "MUNIDESC" = 'Pittsburgh'
  AND "FAIRMARKETTOTAL" > 0
  AND "NEIGHDESC" IS NOT NULL
GROUP BY "NEIGHDESC"
ORDER BY median_value DESC
LIMIT 50

```

**Result preview:**
```
SQL: 
SELECT 
  "NEIGHDESC" AS neighborhood,
  COUNT(*) AS total_parcels,
  ROUND(AVG("FAIRMARKETTOTAL")::numeric, 0) AS avg_fair_market_value,
  ROUND(MIN("FAIRMARKETTOTAL")::numeric, 0) AS min_value,
  ROUND(MAX("FAIRMARKETTOTAL")::numeric, 0) AS max_value,
  ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY "FAIRMARKETTOTAL")::numeric, 0) AS median_value,
  SUM("FAIRMARKETTOTAL") AS total_value
FROM "65855e14-549e-4992-b5be-d629afc676fa"
WHERE "MUNIDESC" = 'Pittsburgh'
  AND "FAIRMARKETTOTAL" > 0
  AND "NEIGHDESC" IS NOT NULL
GROUP BY "NEIGHDESC"
ORDER BY median_value DESC
LIMIT 50

Rows: 0
```


In [ ]:
# Step 4: SQL Analysis Query

# Run SQL analysis query
sql = '\nSELECT \n  "NEIGHDESC" AS neighborhood,\n  COUNT(*) AS total_parcels,\n  ROUND(AVG("FAIRMARKETTOTAL")::numeric, 0) AS avg_fair_market_value,\n  ROUND(MIN("FAIRMARKETTOTAL")::numeric, 0) AS min_value,\n  ROUND(MAX("FAIRMARKETTOTAL")::numeric, 0) AS max_value,\n  ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY "FAIRMARKETTOTAL")::numeric, 0) AS median_value,\n  SUM("FAIRMARKETTOTAL") AS total_value\nFROM "65855e14-549e-4992-b5be-d629afc676fa"\nWHERE "MUNIDESC" = \'Pittsburgh\'\n  AND "FAIRMARKETTOTAL" > 0\n  AND "NEIGHDESC" IS NOT NULL\nGROUP BY "NEIGHDESC"\nORDER BY median_value DESC\nLIMIT 50\n'

resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search_sql", json={"sql": sql})
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Query returned {len(df)} rows")
df


## Step 5: SQL Analysis Query

**SQL:**
```sql

SELECT DISTINCT "MUNIDESC", COUNT(*) as cnt
FROM "65855e14-549e-4992-b5be-d629afc676fa"
WHERE "MUNIDESC" ILIKE '%pittsburgh%'
GROUP BY "MUNIDESC"
ORDER BY cnt DESC
LIMIT 20

```

**Result preview:**
```
SQL: 
SELECT DISTINCT "MUNIDESC", COUNT(*) as cnt
FROM "65855e14-549e-4992-b5be-d629afc676fa"
WHERE "MUNIDESC" ILIKE '%pittsburgh%'
GROUP BY "MUNIDESC"
ORDER BY cnt DESC
LIMIT 20

Rows: 20
Columns: MUNIDESC, cnt

              MUNIDESC   cnt
19th Ward - PITTSBURGH 14047
14th Ward - PITTSBURGH 11507
20th Ward - PITTSBURGH  8504
15th Ward - PITTSBURGH  7188
10th Ward - PITTSBURGH  7011
26th Ward - PITTSBURGH  6323
27th Ward - PITTSBURGH  5966
13th Ward - PITTSBURGH  5963
12th Ward - PITTSBURGH  5050
18th Ward - PITTSBURGH  4947
28th Ward - PITTSBURGH  4757
29th Ward - PITTSBURGH  4610
 4th Ward 
```


In [ ]:
# Step 5: SQL Analysis Query

# Run SQL analysis query
sql = '\nSELECT DISTINCT "MUNIDESC", COUNT(*) as cnt\nFROM "65855e14-549e-4992-b5be-d629afc676fa"\nWHERE "MUNIDESC" ILIKE \'%pittsburgh%\'\nGROUP BY "MUNIDESC"\nORDER BY cnt DESC\nLIMIT 20\n'

resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search_sql", json={"sql": sql})
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Query returned {len(df)} rows")
df


## Step 6: SQL Analysis Query

**SQL:**
```sql

SELECT 
  "NEIGHDESC" AS neighborhood,
  COUNT(*) AS total_parcels,
  ROUND(AVG("FAIRMARKETTOTAL")::numeric, 0) AS avg_fair_market_value,
  ROUND(MIN("FAIRMARKETTOTAL")::numeric, 0) AS min_value,
  ROUND(MAX("FAIRMARKETTOTAL")::numeric, 0) AS max_value,
  ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY "FAIRMARKETTOTAL")::numeric, 0) AS median_value,
  SUM("FAIRMARKETTOTAL") AS total_value
FROM "65855e14-549e-4992-b5be-d629afc676fa"
WHERE "MUNIDESC" ILIKE '%pittsburgh%'
  AND "FAIRMARKETTOTAL" > 0
  AND "NEIGHDESC" IS NOT NULL
GROUP BY "NEIGHDESC"
ORDER BY median_value DESC
LIMIT 50

```

**Result preview:**
```
SQL: 
SELECT 
  "NEIGHDESC" AS neighborhood,
  COUNT(*) AS total_parcels,
  ROUND(AVG("FAIRMARKETTOTAL")::numeric, 0) AS avg_fair_market_value,
  ROUND(MIN("FAIRMARKETTOTAL")::numeric, 0) AS min_value,
  ROUND(MAX("FAIRMARKETTOTAL")::numeric, 0) AS max_value,
  ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY "FAIRMARKETTOTAL")::numeric, 0) AS median_value,
  SUM("FAIRMARKETTOTAL") AS total_value
FROM "65855e14-549e-4992-b5be-d629afc676fa"
WHERE "MUNIDESC" ILIKE '%pittsburgh%'
  AND "FAIRMARKETTOTAL" > 0
  AND "NEIGHDESC" IS NOT NULL
GROUP BY "NEIGHDESC"
ORDER BY median_value DESC
LIMIT 50

R
```


In [ ]:
# Step 6: SQL Analysis Query

# Run SQL analysis query
sql = '\nSELECT \n  "NEIGHDESC" AS neighborhood,\n  COUNT(*) AS total_parcels,\n  ROUND(AVG("FAIRMARKETTOTAL")::numeric, 0) AS avg_fair_market_value,\n  ROUND(MIN("FAIRMARKETTOTAL")::numeric, 0) AS min_value,\n  ROUND(MAX("FAIRMARKETTOTAL")::numeric, 0) AS max_value,\n  ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY "FAIRMARKETTOTAL")::numeric, 0) AS median_value,\n  SUM("FAIRMARKETTOTAL") AS total_value\nFROM "65855e14-549e-4992-b5be-d629afc676fa"\nWHERE "MUNIDESC" ILIKE \'%pittsburgh%\'\n  AND "FAIRMARKETTOTAL" > 0\n  AND "NEIGHDESC" IS NOT NULL\nGROUP BY "NEIGHDESC"\nORDER BY median_value DESC\nLIMIT 50\n'

resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search_sql", json={"sql": sql})
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Query returned {len(df)} rows")
df


## Step 7: SQL Analysis Query

**SQL:**
```sql

SELECT 
  "NEIGHDESC" AS neighborhood,
  COUNT(*) AS total_parcels,
  ROUND(AVG("FAIRMARKETTOTAL")::numeric, 0) AS avg_fair_market_value,
  ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY "FAIRMARKETTOTAL")::numeric, 0) AS median_value,
  ROUND(MAX("FAIRMARKETTOTAL")::numeric, 0) AS max_value,
  SUM("FAIRMARKETTOTAL") AS total_value
FROM "65855e14-549e-4992-b5be-d629afc676fa"
WHERE "MUNIDESC" ILIKE '%pittsburgh%'
  AND "FAIRMARKETTOTAL" > 0
  AND "NEIGHDESC" IS NOT NULL
GROUP BY "NEIGHDESC"
ORDER BY median_value ASC
LIMIT 20

```

**Result preview:**
```
SQL: 
SELECT 
  "NEIGHDESC" AS neighborhood,
  COUNT(*) AS total_parcels,
  ROUND(AVG("FAIRMARKETTOTAL")::numeric, 0) AS avg_fair_market_value,
  ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY "FAIRMARKETTOTAL")::numeric, 0) AS median_value,
  ROUND(MAX("FAIRMARKETTOTAL")::numeric, 0) AS max_value,
  SUM("FAIRMARKETTOTAL") AS total_value
FROM "65855e14-549e-4992-b5be-d629afc676fa"
WHERE "MUNIDESC" ILIKE '%pittsburgh%'
  AND "FAIRMARKETTOTAL" > 0
  AND "NEIGHDESC" IS NOT NULL
GROUP BY "NEIGHDESC"
ORDER BY median_value ASC
LIMIT 20

Rows: 20
Columns: neighborhood, total_parcels, avg_fair_mark
```


In [ ]:
# Step 7: SQL Analysis Query

# Run SQL analysis query
sql = '\nSELECT \n  "NEIGHDESC" AS neighborhood,\n  COUNT(*) AS total_parcels,\n  ROUND(AVG("FAIRMARKETTOTAL")::numeric, 0) AS avg_fair_market_value,\n  ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY "FAIRMARKETTOTAL")::numeric, 0) AS median_value,\n  ROUND(MAX("FAIRMARKETTOTAL")::numeric, 0) AS max_value,\n  SUM("FAIRMARKETTOTAL") AS total_value\nFROM "65855e14-549e-4992-b5be-d629afc676fa"\nWHERE "MUNIDESC" ILIKE \'%pittsburgh%\'\n  AND "FAIRMARKETTOTAL" > 0\n  AND "NEIGHDESC" IS NOT NULL\nGROUP BY "NEIGHDESC"\nORDER BY median_value ASC\nLIMIT 20\n'

resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search_sql", json={"sql": sql})
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Query returned {len(df)} rows")
df


## Step 8: SQL Analysis Query

**SQL:**
```sql

SELECT 
  "NEIGHDESC" AS neighborhood,
  COUNT(*) AS total_parcels,
  ROUND(AVG("FAIRMARKETTOTAL")::numeric, 0) AS avg_fair_market_value,
  ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY "FAIRMARKETTOTAL")::numeric, 0) AS median_value,
  ROUND(MAX("FAIRMARKETTOTAL")::numeric, 0) AS max_value,
  SUM("FAIRMARKETTOTAL") AS total_value
FROM "65855e14-549e-4992-b5be-d629afc676fa"
WHERE "MUNIDESC" ILIKE '%pittsburgh%'
  AND "FAIRMARKETTOTAL" > 0
  AND "NEIGHDESC" IS NOT NULL
  AND LENGTH("NEIGHDESC") > 5
  AND "NEIGHDESC" NOT SIMILAR TO '[0-9]+%'
GROUP BY "NEIGHDESC"
ORDER BY total_parcels DESC
LIMIT 30

```

**Result preview:**
```
SQL: 
SELECT 
  "NEIGHDESC" AS neighborhood,
  COUNT(*) AS total_parcels,
  ROUND(AVG("FAIRMARKETTOTAL")::numeric, 0) AS avg_fair_market_value,
  ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY "FAIRMARKETTOTAL")::numeric, 0) AS median_value,
  ROUND(MAX("FAIRMARKETTOTAL")::numeric, 0) AS max_value,
  SUM("FAIRMARKETTOTAL") AS total_value
FROM "65855e14-549e-4992-b5be-d629afc676fa"
WHERE "MUNIDESC" ILIKE '%pittsburgh%'
  AND "FAIRMARKETTOTAL" > 0
  AND "NEIGHDESC" IS NOT NULL
  AND LENGTH("NEIGHDESC") > 5
  AND "NEIGHDESC" NOT SIMILAR TO '[0-9]+%'
GROUP BY "NEIGHDESC"
ORDER BY total_parcels 
```


In [ ]:
# Step 8: SQL Analysis Query

# Run SQL analysis query
sql = '\nSELECT \n  "NEIGHDESC" AS neighborhood,\n  COUNT(*) AS total_parcels,\n  ROUND(AVG("FAIRMARKETTOTAL")::numeric, 0) AS avg_fair_market_value,\n  ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY "FAIRMARKETTOTAL")::numeric, 0) AS median_value,\n  ROUND(MAX("FAIRMARKETTOTAL")::numeric, 0) AS max_value,\n  SUM("FAIRMARKETTOTAL") AS total_value\nFROM "65855e14-549e-4992-b5be-d629afc676fa"\nWHERE "MUNIDESC" ILIKE \'%pittsburgh%\'\n  AND "FAIRMARKETTOTAL" > 0\n  AND "NEIGHDESC" IS NOT NULL\n  AND LENGTH("NEIGHDESC") > 5\n  AND "NEIGHDESC" NOT SIMILAR TO \'[0-9]+%\'\nGROUP BY "NEIGHDESC"\nORDER BY total_parcels DESC\nLIMIT 30\n'

resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search_sql", json={"sql": sql})
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Query returned {len(df)} rows")
df


## Step 9: SQL Analysis Query

**SQL:**
```sql

SELECT 
  "NEIGHDESC" AS neighborhood,
  COUNT(*) AS total_parcels,
  ROUND(AVG("FAIRMARKETTOTAL")::numeric, 0) AS avg_fair_market_value,
  ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY "FAIRMARKETTOTAL")::numeric, 0) AS median_value,
  ROUND(MAX("FAIRMARKETTOTAL")::numeric, 0) AS max_value,
  ROUND(SUM("FAIRMARKETTOTAL")::numeric / 1000000, 1) AS total_value_millions
FROM "65855e14-549e-4992-b5be-d629afc676fa"
WHERE "MUNIDESC" ILIKE '%pittsburgh%'
  AND "FAIRMARKETTOTAL" > 0
  AND "NEIGHDESC" IS NOT NULL
  AND "NEIGHDESC" NOT SIMILAR TO '[0-9]+%'
  AND "NEIGHDESC" NOT ILIKE '%UTILITY%'
  AND "NEIGHDESC" NOT ILIKE '%MOBILE%'
  AND LENGTH("NEIGHDESC") > 5
GROUP BY "NEIGHDESC"
ORDER BY median_value DESC
LIMIT 25

```

**Result preview:**
```
SQL: 
SELECT 
  "NEIGHDESC" AS neighborhood,
  COUNT(*) AS total_parcels,
  ROUND(AVG("FAIRMARKETTOTAL")::numeric, 0) AS avg_fair_market_value,
  ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY "FAIRMARKETTOTAL")::numeric, 0) AS median_value,
  ROUND(MAX("FAIRMARKETTOTAL")::numeric, 0) AS max_value,
  ROUND(SUM("FAIRMARKETTOTAL")::numeric / 1000000, 1) AS total_value_millions
FROM "65855e14-549e-4992-b5be-d629afc676fa"
WHERE "MUNIDESC" ILIKE '%pittsburgh%'
  AND "FAIRMARKETTOTAL" > 0
  AND "NEIGHDESC" IS NOT NULL
  AND "NEIGHDESC" NOT SIMILAR TO '[0-9]+%'
  AND "NEIGHDESC" NOT ILIKE '%UTILIT
```


In [ ]:
# Step 9: SQL Analysis Query

# Run SQL analysis query
sql = '\nSELECT \n  "NEIGHDESC" AS neighborhood,\n  COUNT(*) AS total_parcels,\n  ROUND(AVG("FAIRMARKETTOTAL")::numeric, 0) AS avg_fair_market_value,\n  ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY "FAIRMARKETTOTAL")::numeric, 0) AS median_value,\n  ROUND(MAX("FAIRMARKETTOTAL")::numeric, 0) AS max_value,\n  ROUND(SUM("FAIRMARKETTOTAL")::numeric / 1000000, 1) AS total_value_millions\nFROM "65855e14-549e-4992-b5be-d629afc676fa"\nWHERE "MUNIDESC" ILIKE \'%pittsburgh%\'\n  AND "FAIRMARKETTOTAL" > 0\n  AND "NEIGHDESC" IS NOT NULL\n  AND "NEIGHDESC" NOT SIMILAR TO \'[0-9]+%\'\n  AND "NEIGHDESC" NOT ILIKE \'%UTILITY%\'\n  AND "NEIGHDESC" NOT ILIKE \'%MOBILE%\'\n  AND LENGTH("NEIGHDESC") > 5\nGROUP BY "NEIGHDESC"\nORDER BY median_value DESC\nLIMIT 25\n'

resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search_sql", json={"sql": sql})
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Query returned {len(df)} rows")
df


## Step 10: SQL Analysis Query

**SQL:**
```sql

SELECT 
  "NEIGHDESC" AS neighborhood,
  COUNT(*) AS total_parcels,
  ROUND(AVG("FAIRMARKETTOTAL")::numeric, 0) AS avg_fair_market_value,
  ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY "FAIRMARKETTOTAL")::numeric, 0) AS median_value,
  ROUND(MAX("FAIRMARKETTOTAL")::numeric, 0) AS max_value,
  ROUND(SUM("FAIRMARKETTOTAL")::numeric / 1000000, 1) AS total_value_millions
FROM "65855e14-549e-4992-b5be-d629afc676fa"
WHERE "MUNIDESC" ILIKE '%pittsburgh%'
  AND "FAIRMARKETTOTAL" > 0
  AND "NEIGHDESC" IS NOT NULL
  AND "NEIGHDESC" NOT SIMILAR TO '[0-9]+%'
  AND "NEIGHDESC" NOT ILIKE '%UTILITY%'
  AND "NEIGHDESC" NOT ILIKE '%MOBILE%'
  AND LENGTH("NEIGHDESC") > 5
GROUP BY "NEIGHDESC"
ORDER BY median_value ASC
LIMIT 15

```

**Result preview:**
```
SQL: 
SELECT 
  "NEIGHDESC" AS neighborhood,
  COUNT(*) AS total_parcels,
  ROUND(AVG("FAIRMARKETTOTAL")::numeric, 0) AS avg_fair_market_value,
  ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY "FAIRMARKETTOTAL")::numeric, 0) AS median_value,
  ROUND(MAX("FAIRMARKETTOTAL")::numeric, 0) AS max_value,
  ROUND(SUM("FAIRMARKETTOTAL")::numeric / 1000000, 1) AS total_value_millions
FROM "65855e14-549e-4992-b5be-d629afc676fa"
WHERE "MUNIDESC" ILIKE '%pittsburgh%'
  AND "FAIRMARKETTOTAL" > 0
  AND "NEIGHDESC" IS NOT NULL
  AND "NEIGHDESC" NOT SIMILAR TO '[0-9]+%'
  AND "NEIGHDESC" NOT ILIKE '%UTILIT
```


In [ ]:
# Step 10: SQL Analysis Query

# Run SQL analysis query
sql = '\nSELECT \n  "NEIGHDESC" AS neighborhood,\n  COUNT(*) AS total_parcels,\n  ROUND(AVG("FAIRMARKETTOTAL")::numeric, 0) AS avg_fair_market_value,\n  ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY "FAIRMARKETTOTAL")::numeric, 0) AS median_value,\n  ROUND(MAX("FAIRMARKETTOTAL")::numeric, 0) AS max_value,\n  ROUND(SUM("FAIRMARKETTOTAL")::numeric / 1000000, 1) AS total_value_millions\nFROM "65855e14-549e-4992-b5be-d629afc676fa"\nWHERE "MUNIDESC" ILIKE \'%pittsburgh%\'\n  AND "FAIRMARKETTOTAL" > 0\n  AND "NEIGHDESC" IS NOT NULL\n  AND "NEIGHDESC" NOT SIMILAR TO \'[0-9]+%\'\n  AND "NEIGHDESC" NOT ILIKE \'%UTILITY%\'\n  AND "NEIGHDESC" NOT ILIKE \'%MOBILE%\'\n  AND LENGTH("NEIGHDESC") > 5\nGROUP BY "NEIGHDESC"\nORDER BY median_value ASC\nLIMIT 15\n'

resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search_sql", json={"sql": sql})
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Query returned {len(df)} rows")
df


## Step 11: SQL Analysis Query

**SQL:**
```sql

SELECT 
  COUNT(*) AS total_pittsburgh_parcels,
  ROUND(AVG("FAIRMARKETTOTAL")::numeric, 0) AS citywide_avg,
  ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY "FAIRMARKETTOTAL")::numeric, 0) AS citywide_median,
  ROUND(SUM("FAIRMARKETTOTAL")::numeric / 1000000000, 2) AS total_value_billions,
  COUNT(DISTINCT "NEIGHDESC") AS num_neighborhoods
FROM "65855e14-549e-4992-b5be-d629afc676fa"
WHERE "MUNIDESC" ILIKE '%pittsburgh%'
  AND "FAIRMARKETTOTAL" > 0

```

**Result preview:**
```
SQL: 
SELECT 
  COUNT(*) AS total_pittsburgh_parcels,
  ROUND(AVG("FAIRMARKETTOTAL")::numeric, 0) AS citywide_avg,
  ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY "FAIRMARKETTOTAL")::numeric, 0) AS citywide_median,
  ROUND(SUM("FAIRMARKETTOTAL")::numeric / 1000000000, 2) AS total_value_billions,
  COUNT(DISTINCT "NEIGHDESC") AS num_neighborhoods
FROM "65855e14-549e-4992-b5be-d629afc676fa"
WHERE "MUNIDESC" ILIKE '%pittsburgh%'
  AND "FAIRMARKETTOTAL" > 0

Rows: 1
Columns: total_pittsburgh_parcels, citywide_avg, citywide_median, total_value_billions, num_neighborhoods

 total_pittsburgh_parc
```


In [ ]:
# Step 11: SQL Analysis Query

# Run SQL analysis query
sql = '\nSELECT \n  COUNT(*) AS total_pittsburgh_parcels,\n  ROUND(AVG("FAIRMARKETTOTAL")::numeric, 0) AS citywide_avg,\n  ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY "FAIRMARKETTOTAL")::numeric, 0) AS citywide_median,\n  ROUND(SUM("FAIRMARKETTOTAL")::numeric / 1000000000, 2) AS total_value_billions,\n  COUNT(DISTINCT "NEIGHDESC") AS num_neighborhoods\nFROM "65855e14-549e-4992-b5be-d629afc676fa"\nWHERE "MUNIDESC" ILIKE \'%pittsburgh%\'\n  AND "FAIRMARKETTOTAL" > 0\n'

resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search_sql", json={"sql": sql})
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Query returned {len(df)} rows")
df


# ============================================================
# 📊 RESULTS
# ============================================================

## 💡 Answer

> **Here is a comprehensive breakdown of **property value distribution across Pittsburgh neighborhoods**, based on the **Allegheny County Property Assessments** dataset from the WPRDC portal (2026 data, 142,813 parcels).

---

## 🏙️ Pittsburgh Property Values at a Glance

| Metric | Value |
|---|---|
| **Total Parcels (Pittsburgh)** | 142,813 |
| **Total Fair Market Value** | ~$34.0 billion |
| **Citywide Average Value** | $238,219 |
| **Citywide Median Value** | $62,600 |
| **Distinct Neighborhoods** | 159 |

> The large gap between average ($238K) and median ($63K) signals significant skew — a small number of high-value properties are pulling the mean way up.

---

## 🏆 Top 15 Highest-Value Neighborhoods (by Median Property Value)

| Neighborhood | Parcels | Median Value | Avg Value | Max Value | Total (Millions) |
|---|---|---|---|---|---|
| Metropolitan Shadyside | 42 | $748,350 | $754,562 | $1,296,000 | $31.7M |
| Skyline & Windom Hill | 11 | $605,400 | $619,173 | $706,800 | $6.8M |
| Bellefield | 78 | $456,800 | $489,659 | $1,048,800 | $38.2M |
| Beacon Street | 27 | $455,000 | $457,285 | $665,000 | $12.3M |
| The Strip District | 92 | $439,300 | $368,753 | $1,291,000 | $33.9M |
| Piatt Place | 84 | $400,850 | $546,132 | $2,625,000 | $45.9M |
| Ward 24 – Troy Hill | 89 | $394,100 | $407,839 | $1,050,000 | $36.3M |
| Trimont Plaza & Marmont | 124 | $360,700 | $361,178 | $835,000 | $44.8M |
| Strip Loft | 266 | $338,550 | $355,783 | $1,405,800 | $94.6M |
| **Shadyside** | **2,133** | **$316,300** | **$361,944** | **$2,284,400** | **$772M** |
| Newer Gardens (Muni 114) | 47 | $312,200 | $309,257 | $459,300 | $14.5M |
| The Carlyle | 60 | $274,000 | $271,765 | $447,300 | $16.3M |
| Market Hoose | 54 | $265,200 | $260,467 | $373,000 | $14.1M |
| Post-War Gardens Muni 114 | 89 | $248,700 | $292,869 | $727,700 | $26.1M |
| Keystone Lofts | 35 | $223,000 | $244,786 | $463,800 | $8.6M |

---

## 📉 Lowest-Value Neighborhoods (by Median Property Value)

| Neighborhood | Parcels | Median Value | Avg Value |
|---|---|---|---|
| Ward 21 – Manchester | 930 | $15,650 | $31,786 |
| Bluff District of Pittsburgh | 324 | $20,000 | $35,235 |
| Crawford-Roberts | 868 | $24,100 | $50,418 |
| Ward 26 – Perry South | 2,524 | $24,350 | $36,367 |
| Ward 24 – Spring Garden | 1,008 | $24,600 | $34,927 |
| Ward 25 | 2,792 | $28,100 | $52,178 |
| East Pittsburgh Borough | 661 | $28,600 | $29,262 |
| Ward 26 – Northview Heights | 917 | $31,200 | $35,009 |
| Ward 24 – Spring Hill-City View | 1,284 | $33,200 | $44,299 |

---

## 📊 Key Takeaways

1. **Shadyside is the dominant high-value large neighborhood**, with 2,133 parcels, a median of $316,300, and a combined total of **$772 million** — by far the highest total of any named neighborhood.
2. **A ~20x value gap** exists between the highest (Metropolitan Shadyside, $748K median) and lowest (Manchester, ~$15K median) neighborhoods.
3. **The Strip District has boomed** — its median of $439,300 reflects the condo/loft development wave.
4. **North and West Pittsburgh neighborhoods** (Perry South, Manchester, Spring Garden, Northview Heights) consistently show the lowest median values, often under $35,000.
5. **The city's wealth is highly concentrated**: the top neighborhoods by total value account for a disproportionate share of the $34B total.

---

**Source:** Allegheny County Property Assessments (2026) — WPRDC (data.wprdc.org). The `FAIRMARKETTOTAL` field represents the county's assessed fair market value, not actual sale prices. Some neighborhood codes are numeric/administrative rather than named neighborhoods.**

---

## 🎯 Confidence Assessment

The Data Concierge evaluates the reliability of its answer using multiple factors:


> ⚠️ Confidence score not available for this query.


# ============================================================
# 📚 CITATIONS & REFERENCES
# ============================================================

## 📖 Data Sources

## Data Sources and Citations

**[1]** Western PA Regional Data Center (WPRDC)
- Dataset: Open Data Portal
- URL: [https://data.wprdc.org](https://data.wprdc.org)
- Accessed: 2026-05-12

---

## 🔄 Reproducibility Guide

This notebook was automatically generated by the ** AI Data Concierge**.
Follow these steps to reproduce or extend the analysis:

### Prerequisites

```bash
pip install pandas numpy requests matplotlib seaborn
```

### Running the Notebook

| Step | Action | Notes |
|------|--------|-------|
| 1 | **Open in Colab** | Click the "Open in Colab" badge at the top |
| 2 | **Run All Cells** | `Runtime` → `Run all` or `Ctrl+F9` |
| 3 | **Wait for completion** | Dependencies install automatically in Colab |
| 4 | **Review results** | Scroll down to see the analysis results |

### ⚠️ Important Notes

- **Data freshness**: Results may differ if data sources have been updated since generation
- **API limits**: Some data sources have rate limits; wait if you encounter errors
- **Modifications**: Feel free to modify parameters and re-run cells to explore further

### 📅 Generation Info

- **Generated**: 2026-05-12 16:26:23
- **Query**: What is the distribution of property values across Pittsburgh neighborhoods?
- **Data Source**: WPRDC

---

*Generated by  AI Data Concierge v0.1.0*
